# Testing Kwok Orchestration with Ko2Cube

This notebook demonstrates how to spin up a Kwok cluster, add nodes (Spot and On-Demand), deploy applications using native Deployments smoothly managed by the Kuberenetes kube-controller-manager, and simulate failures and spot interruptions.

In [13]:
%load_ext autoreload
%autoreload 2

import sys
import os
sys.path.append(os.path.abspath('..'))

from server.kwok.cluster import Cluster

# 1. Create a simulated cluster
cluster = Cluster(name="kwok-test", region="test")


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [18]:
cluster.create()

Creating cluster 'kwok-test' (region=test, port=34099)...
Executing: kwokctl create cluster --name kwok-test --kube-apiserver-port 34099


{"time":"2026-04-06T00:48:36.214803+05:30","level":"INFO","source":{"function":"sigs.k8s.io/kwok/pkg/kwokctl/cmd/create/cluster.runE","file":"/private/tmp/kwok-20251226-5041-tfsm21/kwok-0.7.0/pkg/kwokctl/cmd/create/cluster/cluster.go","line":304},"msg":"Cluster is creating","cluster":"kwok-test"}
{"time":"2026-04-06T00:48:36.733875+05:30","level":"INFO","source":{"function":"sigs.k8s.io/kwok/pkg/kwokctl/cmd/create/cluster.runE","file":"/private/tmp/kwok-20251226-5041-tfsm21/kwok-0.7.0/pkg/kwokctl/cmd/create/cluster/cluster.go","line":311},"msg":"Cluster is created","cluster":"kwok-test","elapsed":{"nanosecond":519069792,"human":"519.069792ms"}}
{"time":"2026-04-06T00:48:36.749087+05:30","level":"INFO","source":{"function":"sigs.k8s.io/kwok/pkg/kwokctl/cmd/create/cluster.runE","file":"/private/tmp/kwok-20251226-5041-tfsm21/kwok-0.7.0/pkg/kwokctl/cmd/create/cluster/cluster.go","line":344},"msg":"Cluster is starting","cluster":"kwok-test"}


Cluster 'kwok-test' created successfully!


{"time":"2026-04-06T00:48:37.262265+05:30","level":"INFO","source":{"function":"sigs.k8s.io/kwok/pkg/kwokctl/cmd/create/cluster.runE","file":"/private/tmp/kwok-20251226-5041-tfsm21/kwok-0.7.0/pkg/kwokctl/cmd/create/cluster/cluster.go","line":349},"msg":"Cluster is started","cluster":"kwok-test","elapsed":{"nanosecond":513161875,"human":"513.161875ms"}}


In [4]:
from server.kwok.config import load_kwok_kubeconfig
load_kwok_kubeconfig("demo-cluster")

Loaded kubeconfig for cluster 'demo-cluster' from dict.


{'apiVersion': 'v1',
 'clusters': [{'cluster': {'certificate-authority-data': 'LS0tLS1CRUdJTiBDRVJUSUZJQ0FURS0tLS0tCk1JSUM0ekNDQWN1Z0F3SUJBZ0lCQURBTkJna3Foa2lHOXcwQkFRc0ZBREFTTVJBd0RnWURWUVFERXdkcmQyOXIKTFdOaE1DQVhEVEkyTURRd05ERTRNemsxT1ZvWUR6SXhNall3TXpFeU1UZ3pPVFU1V2pBU01SQXdEZ1lEVlFRRApFd2RyZDI5ckxXTmhNSUlCSWpBTkJna3Foa2lHOXcwQkFRRUZBQU9DQVE4QU1JSUJDZ0tDQVFFQXV4UzBmNjdZCnUvU2dTUGNaWjgvK3dVekxBNm42T2I0TCtSWWZhTXptT2dZRDg4QmxXUnZYUXlHQlo5RDVGSlpGSTFWQzZjRDkKVWNyRXp6TExnVTErNjROR0Z0dk9WVWNESnFUUXo3RG9jaWp5bERnL0UrczNSY3hkOU9HK0lyYWRaY3FLdG5OQgpaTUJuMEFCNTRYdjBwbmZwc2h6SWhtS2o3S082WkhONHRlSGZPSnNCTytFM0JLSnc3REN6ZGUreVBaWUMvNDhzCkhGTWl6Ykg2UlhFV0R1VVBxMXNsaU9YZnlHb24vWWRiMk90c1V1ejA2VkQreHcrNVFTSlNtcGtEOGZ5T1Bzb0YKT0JwSTJpaFRmbExwRWo5bzVrK0hYaFdNdEJRT3daQW5CaGZEa0FKaGRIVVZqQ0FXNnNyeEpPU1premNKZnoyQgpxL0NlalM3NDRvK2NSd0lEQVFBQm8wSXdRREFPQmdOVkhROEJBZjhFQkFNQ0FxUXdEd1lEVlIwVEFRSC9CQVV3CkF3RUIvekFkQmdOVkhRNEVGZ1FVZ2t2b0tUSVpQaitMWkVNazN3TE9HS0ZOREhRd0RRWUpLb1pJaHZjTkFRRUwKQlFBRGdnRUJBRERRb

In [22]:
pod=cluster.add_pod("test1","node-main")

Loaded kubeconfig for cluster 'kwok-test' from dict.
Creating pod 'test1' on node 'node-main'...
Pod 'test1' created successfully!


In [25]:
pod.complete()

Loaded kubeconfig for cluster 'kwok-test' from dict.
Pod 'test1' status changed to Succeeded


In [7]:
cluster.get_pods()

Loaded kubeconfig for cluster 'demo-cluster' from dict.


[{'name': 'test1',
  'namespace': 'default',
  'phase': 'Pending',
  'node_name': 'node-main',
  'containers': ['fake-container']}]

In [19]:
# 2. Spawn a Mix of On-Demand and Spot Nodes
node_regular = cluster.add_node(name="node-main", instance_type="m5.large", capacity_type="ON_DEMAND")
node_spot = cluster.add_node(name="node-spot-1", instance_type="c5.xlarge", capacity_type="SPOT")

print("Tracked python node references in Cluster:", list(cluster.kwok_nodes.keys()))

# Verify from Kubernetes API
for n in cluster.get_nodes():
    print(f"API Node Name: {n['name']}, Region: {n['region']}, Instance: {n['instance_type']}")

Loaded kubeconfig for cluster 'kwok-test' from dict.
Creating node 'node-main' (m5.large, test)...
{'apiVersion': 'v1', 'kind': 'Node', 'metadata': {'name': 'node-main', 'annotations': {'node.kubernetes.io/ttl': '0', 'kwok.x-k8s.io/node': 'fake'}, 'labels': {'kubernetes.io/arch': 'amd64', 'kubernetes.io/hostname': 'node-main', 'kubernetes.io/os': 'linux', 'kubernetes.io/role': 'agent', 'node-role.kubernetes.io/agent': '', 'node.kubernetes.io/instance-type': 'm5.large', 'topology.kubernetes.io/region': 'test', 'eks.amazonaws.com/capacityType': 'ON_DEMAND', 'type': 'kwok'}}, 'spec': {'taints': [{'effect': 'NoSchedule', 'key': 'kwok.x-k8s.io/node', 'value': 'fake'}]}, 'status': {'capacity': {'cpu': '2', 'memory': '8Gi', 'pods': '29'}, 'allocatable': {'cpu': '2', 'memory': '8Gi', 'pods': '29'}, 'nodeInfo': {'architecture': 'amd64', 'bootID': '', 'containerRuntimeVersion': '', 'kernelVersion': '', 'kubeProxyVersion': 'fake', 'kubeletVersion': 'fake', 'machineID': '', 'operatingSystem': 'lin

{'apiVersion': 'v1', 'kind': 'Node', 'metadata': {'name': 'node-main', 'annotations': {'node.kubernetes.io/ttl': '0', 'kwok.x-k8s.io/node': 'fake'}, 'labels': {'kubernetes.io/arch': 'amd64', 'kubernetes.io/hostname': 'node-main', 'kubernetes.io/os': 'linux', 'node.kubernetes.io/instance-type': 'm5.large', 'topology.kubernetes.io/region': 'test', 'eks.amazonaws.com/capacityType': 'ON_DEMAND', 'type': 'kwok'}}, 'spec': {'taints': [{'effect': 'NoSchedule', 'key': 'kwok.x-k8s.io/node', 'value': 'fake'}]}, 'status': {'capacity': {'cpu': '2', 'memory': '8Gi', 'pods': '29'}, 'allocatable': {'cpu': '2', 'memory': '8Gi', 'pods': '29'}, 'phase': 'Running'}}


In [ ]:
# 3. Deploy an App using Native Deployments
deployment = cluster.add_deployment(
    name="ai-agent-service", 
    replicas=3, 
    image="agent-image:v1"
)

import time
print("Waiting 3 seconds for kube-controller-manager to spin up the ReplicaSet...")
time.sleep(3) 

print("\nTracked Deployments:", list(cluster.kwok_deployments.keys()))
pods = cluster.get_pods()
print(f"\nTotal Pods Running in API: {len(pods)}")
for p in pods:
    print(f"- {p['name']} on {p['node_name']} (Phase: {p['phase']})")

In [ ]:
# 4. Simulate a Rollout update
print("Patching deployment image to v2...")
deployment.apply_new_rollout("agent-image:v2")
print("\nWait a moment, then check cluster.get_pods() below to see the pods replaced natively!")

In [ ]:
time.sleep(5)
for p in cluster.get_pods():
    print(f"- {p['name']} on {p['node_name']} (Phase: {p['phase']})")

In [ ]:
# 5. Simulate Spot Interruption
def my_callback():
    print("\n*** AGENT CALLBACK FIRED: Spot Node Interrupted and Deleted! ***\n")

node_spot.simulate_spot_interruption(delay_seconds=2, callback=my_callback)


In [ ]:
# Verify that the Python Cluster state properly evicted the Spot node via the deletion callback
time.sleep(3)
print("\nRemaining Tracked Nodes in Python:", list(cluster.kwok_nodes.keys()))

In [ ]:
# Create a parallelized batch job
job = cluster.add_job(
    name="ai-training-job",
    node_name="node-main",               # The node you want it to run on
    completions=5,                       # We need 5 total pod completions
    parallelism=2,                       # Run 2 pods concurrently
    cpu="500m",
    duration="30s",   # Pod stays Running for 30 s                           
    memory="512Mi",
    backoff_limit=2                      # Fail the job entirely if 2 pods crash
)

import time
time.sleep(2) # Give kube-controller a second to spin them up

# Verify via Python tracking:
print("Tracked Jobs:", list(cluster.kwok_jobs.keys()))

# Verify via your Python Kubernetes wrapper:
for j in cluster.get_jobs():
    print(f"Job Name: {j['name']} | Status: {j['succeeded']}/{j['completions']} completed")


Loaded kubeconfig for cluster 'kwok-test' from dict.
Creating job 'ai-training-job' on node 'node-main'...
Job 'ai-training-job' created successfully!
Tracked Jobs: ['ai-training-job']
Loaded kubeconfig for cluster 'kwok-test' from dict.
Job Name: ai-training-job | Status: 2/5 completed


In [ ]:
for j in cluster.get_jobs():
    print(j)
    print(f"Job Name: {j['name']} | Status: {j['succeeded']}/{j['completions']} completed")

In [3]:
# 6. Cleanup
cluster.delete()

Deleting cluster 'demo-cluster'...


{"time":"2026-04-06T00:39:25.519596+05:30","level":"INFO","source":{"function":"sigs.k8s.io/kwok/pkg/kwokctl/cmd/delete/cluster.deleteCluster","file":"/private/tmp/kwok-20251226-5041-tfsm21/kwok-0.7.0/pkg/kwokctl/cmd/delete/cluster/cluster.go","line":121},"msg":"Cluster is stopping","cluster":"demo-cluster"}
{"time":"2026-04-06T00:39:25.904226+05:30","level":"INFO","source":{"function":"sigs.k8s.io/kwok/pkg/kwokctl/cmd/delete/cluster.deleteCluster","file":"/private/tmp/kwok-20251226-5041-tfsm21/kwok-0.7.0/pkg/kwokctl/cmd/delete/cluster/cluster.go","line":126},"msg":"Cluster is stopped","cluster":"demo-cluster","elapsed":{"nanosecond":385456250,"human":"385.45625ms"}}
{"time":"2026-04-06T00:39:25.904328+05:30","level":"INFO","source":{"function":"sigs.k8s.io/kwok/pkg/kwokctl/cmd/delete/cluster.deleteCluster","file":"/private/tmp/kwok-20251226-5041-tfsm21/kwok-0.7.0/pkg/kwokctl/cmd/delete/cluster/cluster.go","line":132},"msg":"Cluster is deleting","cluster":"demo-cluster"}


Cluster 'demo-cluster' deleted.


{"time":"2026-04-06T00:39:26.599758+05:30","level":"INFO","source":{"function":"sigs.k8s.io/kwok/pkg/kwokctl/cmd/delete/cluster.deleteCluster","file":"/private/tmp/kwok-20251226-5041-tfsm21/kwok-0.7.0/pkg/kwokctl/cmd/delete/cluster/cluster.go","line":148},"msg":"Cluster is deleted","cluster":"demo-cluster","elapsed":{"nanosecond":695421500,"human":"695.4215ms"}}
